In [1]:
import sys
import os

# Navigate up from notebooks/ to main/, then add explainers/ to the path
current_dir = os.getcwd()  # Returns main/notebooks/
parent_dir = os.path.dirname(current_dir)  # Returns main/
sys.path.append(parent_dir)  # Now Python can see main/

In [2]:
# Activate Conda Env "Ultra" As the Kernel

import torch
import pandas as pd
from explainers.data_util import batched_k_hop_subgraph
import networkx as nx
from pyvis.network import Network

In [8]:
dataset = torch.load("/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/misc/nell995_dataset.pt", map_location=torch.device('cpu'))
id2entity, id2relation = torch.load("/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/misc/nell995_id2name.pt", map_location=torch.device('cpu'))

In [10]:
split = 'valid' # the split
ratio = 100 # the top r ratio of edges

if split == 'valid':
    data_index = 1
if split == 'test':
    data_index = 2

output_file_root="/scratch/sl9030/gml/Graph-Transformer/NBFNet-PyG/explanation/RAWExplainer/NBFNet/NELL995/2025-10-01-16-28-28-save-expl-optim"

# The path of dataset
output_file = f"{output_file_root}/{split}_output_factual_eval_hard_edge_mask_top_k_{ratio}.pt"
explanation_file = f"{output_file_root}/{split}_explanations_factual_eval_hard_edge_mask_top_k_{ratio}.pt"

outputs = torch.load(output_file, map_location=torch.device('cpu'))
explanations = torch.load(explanation_file, map_location=torch.device('cpu'))
output_df = pd.DataFrame(outputs)

data = dataset[data_index]

In [11]:
output_df.iloc[1]

Ranking          2
Heads        46484
Tails         3872
Rel            101
Mode             1
Num_Edges       92
Num_Nodes       24
Inclusion        1
Name: 1, dtype: int64

In [14]:
explanations[0]

tensor([ 24832,     60,   3165,   3739,  13874,  14545,  15753,  15896,  20660,
         21915,  22636,  23083,  28300,  40277,  41234,  41932,  43079,  43470,
         45153,  55077,  59620,  60202,  62077,  62759,  77187,  79454,  80121,
         81281,  82329,  84810,  85926,  87495,  91295,  93909,  98446, 102219,
        102588, 109866, 111904, 120807, 121380, 121567, 131127, 131143, 143017,
        143759, 145868, 148719, 152843, 153417, 163552, 164223, 165431, 165574,
        170338, 171593, 172314, 172761, 177978, 189955, 190912, 191610, 192757,
        193148, 194831, 204755, 209298, 209880, 211755, 212437, 226865, 229132,
        229799, 230959, 232007, 234488, 235604, 237173, 240973, 243587, 248124,
        251897, 252266, 259544, 261582, 270485, 271058, 271245, 280805, 280821,
        292695, 293437, 295546, 298397,     -1,     -1,     -1,     -1,     -1,
            -1,     -1,     -1], dtype=torch.int32)

In [12]:
explanations[1].sum()

tensor(13564711)

In [15]:
def vizualize_explanation(index):
    row = output_df.iloc[index]
    if row['Mode'] == 1:
        rel = id2relation[row['Rel']]
    else:
        rel = id2relation[row['Rel']]+'_inv'

    head_id = row['Heads']
    tail_id = row['Tails']
    head = id2entity[head_id]
    tail = id2entity[tail_id]

    print(f"*** Query: {head, rel}. Answer: {tail}, Rank given the explanation: {row['Ranking']} ***")
    nodes, edges = batched_k_hop_subgraph(torch.tensor([head_id]).unsqueeze(0), 6, data.edge_index, data.num_nodes)

    print(f'Is the tail included in the 6-hop neighbor of head in the original graph? {nodes[0, tail_id].item()}')
    full_edge_index = data.edge_index[:, edges.squeeze()]

    expl = explanations[index]
    # Check that the explanation are restricted only to within the ego_network (not necessary)
    # assert torch.all(torch.logical_and(expl, ~edges.squeeze()) == False)
    expl_edge_index = data.edge_index[:, expl.to(torch.bool)]

    print(f'Is the head included in the Explanation? {torch.any((expl_edge_index == head_id)).item()}')
    print(f'Is the tail included in the Explanation? {torch.any((expl_edge_index == tail_id)).item()}')

    # Create a directed graph
    G = nx.DiGraph()

    # Add edges from expl_edge_index
    for i in range(expl_edge_index.shape[1]):
        source = expl_edge_index[0, i].item()
        target = expl_edge_index[1, i].item()
        # G.add_edge(source, target,arrows='to')
        G.add_edge(source, target)

    attrs = {head_id: "#FF0000", tail_id: "#00FF00"}
    nx.set_node_attributes(G, attrs, name='color')

    try:
        paths = list(nx.all_simple_paths(G, source = head_id, target=tail_id, cutoff = 6))
    
        attrs = {}
        if len(paths) > 0:
            connected = True
        else:
            connected = False
    
        print(f'Is the tail connected to the head? {connected}')
        for path in paths:
            src = head_id
            for node in path[1:]:
                edge = (src, node)
                attrs[edge] = {'color':"#b27ebd"}
                src = node

        nx.set_edge_attributes(G, attrs)
    except:
        print(f'Is the tail connected to the head? {False}')
            
    
    return G
    

In [16]:
display(output_df)

,Ranking,Heads,Tails,Rel,Mode,Num_Edges,Num_Nodes,Inclusion
0,3,24832,4274,60,1,92,12,1
1,2,46484,3872,101,1,92,24,1
2,5,34770,8189,38,1,94,22,1
3,1,73893,55955,74,1,100,18,1
4,74532,4274,24832,60,0,92,17,0
...,...,...,...,...,...,...,...,...
1081,6,33551,1657,60,1,84,18,1
1082,3,45452,438,107,1,82,10,1
1083,74317,31,40614,61,0,92,24,0
1084,74525,1657,33551,60,0,98,20,0


In [17]:
# Control the index (row number of output_df of the instance you want to inspect here)
index = 3

In [ ]:
G = vizualize_explanation(index)

In [27]:
# *** Explanation Graph Visualization ***
# RED NODE: The Query Head (if inside the explanation)
# GREEN NODE: The Answer Tail (if inside the explanation)
# Purple Edges: Edges that are in any path that connected Red to Green
net = Network(notebook = True, cdn_resources = "remote",
                    bgcolor = "#222222",
                    font_color = "white",
                    height = "750px",
                    width = "100%",
                    select_menu = True,
                    filter_menu = True,
                    directed=True
    )
net.from_nx(G)
net.inherit_edge_colors(False)
net.set_edge_smooth('dynamic')
net.show('graph.html')

graph.html
